# 03 · Bayesian optimization on LCRC Crossover — result analysis

Workflow 3 maximizes `profit` over the four knobs with a GP + qLogEI loop:
16 LHS warm-up samples, then batches of 4, every sample running as its own
Slurm job on Crossover. The study is submitted and monitored **from the CLI on
a login node** — that's the natural interface on a cluster:

```bash
polarisopt validate studies/bo-crossover.yaml
polarisopt plan     studies/bo-crossover.yaml   # render sbatch, submit nothing
polarisopt run      studies/bo-crossover.yaml   # blocks while the loop runs

# from a second shell
polarisopt status studies/bo-crossover.yaml
polarisopt logs   studies/bo-crossover.yaml <SAMPLE_ID>
polarisopt retry-failed studies/bo-crossover.yaml --run
```

This notebook only *reads* the finished study. Run it anywhere that can see
the workspace (a Crossover login node, or copy `polarisopt.db` locally).

In [ ]:
import os
from pathlib import Path

WORKSPACE = Path(f"/lcrc/project/POLARIS/{os.environ['USER']}/polarisopt-runs/taxidemo-bo")
assert WORKSPACE.exists(), "run `polarisopt run studies/bo-crossover.yaml` first (or point WORKSPACE at a copy)"


In [ ]:
import pandas as pd
from polarisopt.samples.store import SampleStore
from polarisopt.utils.paths import workspace_layout

layout = workspace_layout(WORKSPACE)
store = SampleStore.open(layout["db"], "taxi-bo")
raw = store.to_dataframe()

PARAMS = ["taxi_count", "base_fare", "cost_per_tile", "max_multiplier"]
df = pd.concat(
    [
        raw[["id", "iteration", "status", "runtime_s"]],
        pd.DataFrame(raw["inputs"].tolist(), columns=PARAMS, index=raw.index),
    ],
    axis=1,
)
df["profit"] = [m[0] if m else None for m in raw["metric"]]
df = df[df["status"] == "finished"].sort_values("id").reset_index(drop=True)
print(f"{len(df)} finished samples, {df['iteration'].max()} BO iterations")
df.tail()

## Convergence

Best profit found so far vs evaluation number. Iteration 0 is the LHS warm-up;
after it the acquisition function takes over and improvements come in batches
of 4.

In [ ]:
import matplotlib.pyplot as plt

n_warmup = (df["iteration"] == 0).sum()
evals = range(1, len(df) + 1)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(evals, df["profit"].cummax(), drawstyle="steps-post", color="k", lw=2, label="best so far")
sc = ax.scatter(evals, df["profit"], c=df["iteration"], cmap="viridis", zorder=3, label=None)
ax.axvline(n_warmup + 0.5, ls="--", c="gray")
ax.text(n_warmup + 1, df["profit"].min(), " warm-up → BO", color="gray")
ax.set_xlabel("evaluation")
ax.set_ylabel("profit")
ax.legend(loc="lower right")
fig.colorbar(sc, ax=ax, label="iteration");

## Where did BO spend its budget?

Warm-up samples (squares) are spread uniformly; BO samples (circles, brighter =
later) cluster in the profitable region — moderate-to-large fleet, prices kept
below the customer-cancellation cliff.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
warm = df["iteration"] == 0
for ax, (xp, yp) in zip(axes, [("taxi_count", "base_fare"), ("cost_per_tile", "max_multiplier")]):
    ax.scatter(df.loc[warm, xp], df.loc[warm, yp], marker="s", c="lightgray", edgecolor="k", label="warm-up")
    sc = ax.scatter(df.loc[~warm, xp], df.loc[~warm, yp], c=df.loc[~warm, "iteration"], cmap="viridis", label="BO")
    ax.set_xlabel(xp)
    ax.set_ylabel(yp)
axes[0].legend()
fig.colorbar(sc, ax=axes, label="iteration", pad=0.01);

## Best design

In [ ]:
best = df.loc[df["profit"].idxmax()]
print(f"best profit: {best['profit']:,.0f}  (sample id {int(best['id'])}, iteration {int(best['iteration'])})")
best[PARAMS].astype(float).round(2)

## BO vs the local LHS screen

If you ran workflow 2 on this machine, compare what 64 uniform samples found
against what the sequential loop found with a similar budget.

In [ ]:
lhs_db = Path(os.path.expanduser("~/taxidemo-runs/lhs-local")) / "polarisopt.db"
if lhs_db.exists():
    lhs_store = SampleStore.open(lhs_db, "taxi-lhs")
    lhs_raw = lhs_store.to_dataframe()
    lhs_best = max(m[0] for m in lhs_raw["metric"] if m)
    print(f"LHS screen best (n={len(lhs_raw)}):  {lhs_best:,.0f}")
    print(f"BO best         (n={len(df)}):  {df['profit'].max():,.0f}")
else:
    print("local LHS study not found — run workflow 2 first for the comparison")